# M5: WildGuardMix topic map and its human check (D-024, D-043, D-086)

**Notebook settings:** Accelerator **GPU T4 x2** (step `map`; step `agreement` also runs on CPU) · Internet **on** · Secrets: **HF_TOKEN**. Run as **Save Version → Save & Run All**.

Set `STEP` in the first code cell:
- **`map`:** Qwen2.5-7B-Instruct (vLLM, TP=2, greedy) assigns each WildGuardMix-test social-stereotype prompt one of the nine topics or `none`. It writes `artifacts/topic_map/{topic_map.csv, labelling_sheet.csv, map_summary.json}` and syncs them to the private store.
  - `topic_map.csv` holds prompt SHA-256 hashes and topic names only, so it can go to git.
  - `labelling_sheet.csv` holds the prompt text and **no** predicted topic (blind labelling). It stays private: download it from this version's **Output** tab only.
- **`agreement`:** label the sheet offline (fill the `label` column with one of the allowed labels in the `#` row). Upload the filled sheet as a **private** Kaggle dataset, attach it, and set `LABELS`. The step restores the map from the store and reports agreement. The map is accepted at ≥ 80% (D-043).

Only counts and hashes are printed.

In [ ]:
REPO = "SGupta-4/Reference-Based-Bias-Detection-Implementation"
REF = "main"  # a branch, a tag, or a FULL 40-character commit SHA (short SHAs fail)
WORK = "/kaggle/working/rbbd"
STEP = "map"  # "map" first; "agreement" after labelling the sheet
LABELS = "/kaggle/input/rbbd-topic-labels/labelling_sheet.csv"  # the filled sheet (private dataset); step "agreement" only
import time
SESSION_T0 = time.time()

In [ ]:
# Clone the repo at REF (branch, tag or commit SHA). If the repo is private, attach the
# GITHUB_TOKEN secret: it is passed to git as an HTTP header through environment
# variables, never in the URL, argv or output.
import base64, datetime, os, shutil, subprocess
from kaggle_secrets import UserSecretsClient

os.environ.setdefault(
    "RBBD_SESSION_ID", datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)
RBBD_SESSION_ID = os.environ["RBBD_SESSION_ID"]  # also expands $RBBD_SESSION_ID in ! lines
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # fail instead of prompting for a password
try:
    _gh = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    _gh = None
if _gh:
    _basic = base64.b64encode(f"x-access-token:{_gh}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {_basic}")
del _gh

shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
for args in (["init", "-q"], ["remote", "add", "origin", f"https://github.com/{REPO}.git"],
             ["fetch", "-q", "--depth", "1", "origin", REF], ["checkout", "-q", "FETCH_HEAD"]):
    done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode and args[0] == "fetch" and "GIT_CONFIG_COUNT" in git_env:
        # An expired or revoked token gets HTTP 401 even on a public repo (B-025): retry
        # without it. git's stderr never contains the header value.
        print("GITHUB_TOKEN was rejected (expired or revoked?); retrying without it")
        git_env = {k: v for k, v in git_env.items() if not k.startswith("GIT_CONFIG_")}
        done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(f"git {args[0]} failed: {done.stderr[-500:]}")
del git_env
os.chdir(WORK)
print("commit", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
      "| session", os.environ["RBBD_SESSION_ID"])

In [ ]:
# B-032 / D-087: the Kaggle image now runs Python 3.13 and vllm 0.10.1.1 has no 3.13 wheels.
# The pinned stack goes into a Python 3.12 venv on ephemeral disk; putting its bin/ first on
# PATH makes every `!python ...` line (and the stage subprocesses) use it. This kernel only
# imports rbbd's light modules (config, runner, manifests) from src/.
import os
EPH = next((p for p in ("/kaggle/tmp", "/kaggle/temp") if os.path.isdir(p)), "/tmp")
VENV = f"{EPH}/rbbd_py312"
os.environ.update(UV_CACHE_DIR=f"{EPH}/uv_cache", UV_LINK_MODE="copy")
!pip install -q uv==0.8.17 2>&1 | tail -2
!uv venv -q --python 3.12 {VENV}
os.environ["VIRTUAL_ENV"] = VENV
os.environ["PATH"] = f"{VENV}/bin:" + os.environ["PATH"]
!uv pip install -q -e ".[train,bench,dev]" 2>&1 | tail -5
!uv pip freeze > /kaggle/working/pip_freeze_m5_topic_map.txt
# Preflight (B-012, D-049): stop here, not inside a stage, if the pinned stack did not install.
import subprocess
_pf = subprocess.run(["python", "-c", "import sys, torch, torchvision, transformers, vllm; "
                      "print('preflight ok', sys.version.split()[0], sys.executable, torch.__version__, "
                      "torchvision.__version__, transformers.__version__, vllm.__version__)"],
                     capture_output=True, text=True)
print(_pf.stdout.strip(), _pf.stderr[-1500:])
if _pf.returncode:
    raise RuntimeError("preflight failed: the Python 3.12 venv stack did not install (see above)")

In [ ]:
# Copy Kaggle Secrets into the environment (presence flags only are printed) and keep the
# HF cache on ephemeral disk so it never counts against /kaggle/working (D-029/D-041).
import sys
sys.path.insert(0, f"{WORK}/src")  # the editable install is not visible to this already-running kernel
from rbbd.utils.env import ephemeral_dir, load_kaggle_secrets

_secrets = load_kaggle_secrets(["HF_TOKEN"])
print(_secrets)
if not _secrets["HF_TOKEN"]:
    # Stop here instead of failing later on every gated download (B-022).
    raise RuntimeError("HF_TOKEN secret not attached: Add-ons -> Secrets -> enable HF_TOKEN "
                       "for this notebook, then run again")
os.environ["HF_HOME"] = str(ephemeral_dir() / "hf")
ENV_DIR = f"/kaggle/working/artifacts/env/{os.environ['RBBD_SESSION_ID']}"
os.makedirs(ENV_DIR, exist_ok=True)

In [ ]:
# Aggregates only: topic counts, agreement, the map's hash. No prompt text is printed.
import json
from pathlib import Path
ROOT = Path("/kaggle/working/artifacts")
TM = ROOT / "topic_map"
if STEP == "map":
    !python -m rbbd.cli topic-map --n-sheet 100
    summary = json.loads((TM / "map_summary.json").read_text())
    import csv
    n_sheet = sum(1 for r in csv.DictReader(open(TM / "labelling_sheet.csv", newline="")) if r["row"] != "#")
    print(json.dumps({**summary, "sheet_rows": n_sheet}, indent=1))
    !python -m rbbd.cli sync --path topic_map
    print("Download artifacts/topic_map/labelling_sheet.csv and topic_map.csv from the Output tab.")
elif STEP == "agreement":
    assert Path(LABELS).exists(), f"attach the filled sheet as a private dataset: {LABELS}"
    !python -m rbbd.cli restore --path topic_map
    !python -m rbbd.cli topic-agreement --labels {LABELS} --map {TM}/topic_map.csv
    !python -m rbbd.cli sync --path topic_map
else:
    raise ValueError(STEP)

In [ ]:
!python -m rbbd.cli sync --path env/$RBBD_SESSION_ID
print("session hours:", round((time.time() - SESSION_T0) / 3600, 2))